# Step 06 Program Gate Overview

End-to-end demonstration of Step 06 (Program & Domain Constraint Engine) on
the 6-fixture matrix:

- **Stage 00** — fixture load + target consistency check.
- **Stage 01** — `ProgramInstance` instantiation + cardinality gate (D004,
  D023 required-only).
- **Stage 02** — Domain Feasibility Gate (D020, D023, D024) — fail-only,
  required-only, gross footprint × density first-pass.

The headline event is **R2 (`apartment_too_small.json`)** finally tripping
`AreaGateFailure`. Plan v2's metadata `expected_failure: AreaGateFailure /
verified_at: Step 06` becomes reality here.

## Output convention (S03-D13)

Generated charts go to
`outputs/notebooks/step06_program_gate_overview/<run_id>/`. Re-runs create
new folders. `outputs/` is gitignored (D014).

## Sections

1. Fixture × Gate matrix
2. Area budget — required vs capacity (the headline)
3. Footprint geometry × Stage 02 result
4. Required-area breakdown by role
5. R2 spotlight — first `AreaGateFailure`
6. D023 demo — required-only cardinality
7. Notes / next step


In [ ]:
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(f"pyproject.toml not found from {start} upward")


ROOT = _find_repo_root(Path.cwd())
print("repo root:", ROOT)


In [ ]:
import sys
from datetime import datetime

if str(ROOT / "tests") not in sys.path:
    sys.path.insert(0, str(ROOT / "tests"))

import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Polygon

from fixture_matrix import MATRIX, fixture_path
from proto3.schema.input import BuildingInput
from proto3.schema.program import ProgramRequest, SpaceUnitSpec
from proto3.schema.serialize import from_json
from proto3.schema.validation import (
    AreaGateFailure,
    DimGateFailure,
    DomainGateFailure,
    ProgramInstantiationFailure,
)
from proto3.stages import stage00_load, stage01_program, stage02_gate
from proto3.target import DEFAULT_APARTMENT_RULES_PATH, TargetAdapter

adapter = TargetAdapter(DEFAULT_APARTMENT_RULES_PATH)
rules = adapter.target_rules()

run_id = datetime.now().strftime("%Y%m%dT%H%M%S")
out_dir = ROOT / "outputs" / "notebooks" / "step06_program_gate_overview" / run_id
out_dir.mkdir(parents=True, exist_ok=True)
print("run output dir:", out_dir.relative_to(ROOT))
print("apartment rules:")
print(f"  target_type:           {rules.target_type}")
print(f"  density_factor:        {rules.density_factor}")
print(f"  requires_single_floor: {rules.requires_single_floor}")
print(f"  min_cardinality:       {rules.min_cardinality}")
print(f"  default_min_area_m2:   {rules.default_min_area_m2}")


In [ ]:
# Run Stage 00 -> Stage 01 -> Stage 02 on every fixture and collect results.

results = {}

for mid in sorted(MATRIX.keys()):
    p = fixture_path(mid)
    building = from_json(BuildingInput, p)

    rec = {
        "matrix_id": mid,
        "file": MATRIX[mid]["file"],
        "expected_failure": MATRIX[mid].get("expected_failure"),
        "building": building,
        "stage01_status": "pass",
        "stage01_failure": None,
        "stage01_failure_type": None,
        "stage02_status": "skip",
        "stage02_failure": None,
        "stage02_failure_type": None,
        "stage02_evidence": None,
        "instance": None,
    }

    try:
        instance = stage01_program.run(building, adapter=adapter)
        rec["instance"] = instance
    except ProgramInstantiationFailure as e:
        rec["stage01_status"] = "fail"
        rec["stage01_failure"] = type(e).__name__
        rec["stage01_failure_type"] = e.failure.failure_type
        results[mid] = rec
        continue

    try:
        stage02_gate.run(building, instance=instance, adapter=adapter)
        rec["stage02_status"] = "pass"
    except DomainGateFailure as e:
        rec["stage02_status"] = "fail"
        rec["stage02_failure"] = type(e).__name__
        rec["stage02_failure_type"] = e.failure.failure_type
        rec["stage02_evidence"] = e.failure.evidence

    results[mid] = rec

print("pipeline run summary:")
print(f"  {'ID':<3}  {'stage01':<7}  {'stage02':<7}  expected")
for mid in sorted(results):
    r = results[mid]
    print(f"  {mid:<3}  {r['stage01_status']:<7}  {r['stage02_status']:<7}  "
          f"{r['expected_failure'] or '(none)'}")


## §1 Fixture × Gate matrix

`pass` / `fail` / `skip` (skip = upstream stage already failed). The
"Actual" column is empty for clean passes.

In [ ]:
from IPython.display import Markdown, display

EMOJI = {"pass": "✓", "fail": "✗", "skip": "—"}

header = "| ID | File | Stage 01 | Stage 02 | Expected | Actual failure |\n"
header += "|---|---|:---:|:---:|---|---|\n"

rows = [header]
for mid in sorted(results):
    r = results[mid]
    if r["stage01_status"] == "fail":
        actual = f"`{r['stage01_failure_type']}` ({r['stage01_failure']})"
    elif r["stage02_status"] == "fail":
        actual = f"`{r['stage02_failure_type']}` ({r['stage02_failure']})"
    else:
        actual = "—"
    rows.append(
        f"| {mid} | `{r['file']}` | {EMOJI[r['stage01_status']]} | "
        f"{EMOJI[r['stage02_status']]} | "
        f"{r['expected_failure'] or '—'} | {actual} |"
    )

display(Markdown("\n".join(rows)))


## §2 Area budget — required vs capacity (the headline)

For each fixture that passes Stage 01, plot the required-area sum vs the
gross footprint × density capacity. Bars exceeding the capacity bar fail
`check_min_area` → `AreaGateFailure`. R2 should be the only red.

In [ ]:
chart_fixtures = [mid for mid in sorted(results)
                  if results[mid]["stage01_status"] != "fail"]

required_areas = []
capacities = []
labels = []
fail_flags = []

for mid in chart_fixtures:
    r = results[mid]
    inst = r["instance"]
    required_sum = sum((u.min_area_m2 or 0)
                       for u in inst.space_units if u.required)
    floor = r["building"].floors[0]
    poly = Polygon(floor.footprint)
    area_m2 = poly.area / 1_000_000.0
    capacity = area_m2 * rules.density_factor

    required_areas.append(required_sum)
    capacities.append(capacity)
    labels.append(f"{mid}\n{r['file']}")
    fail_flags.append(required_sum > capacity)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(labels))
w = 0.36

req_colors = ["#e04a3a" if f else "#4a90d6" for f in fail_flags]
bars_req = ax.bar(x - w / 2, required_areas, w,
                  label="Σ required min_area_m2", color=req_colors,
                  edgecolor="black", lw=0.5)
bars_cap = ax.bar(x + w / 2, capacities, w,
                  label=f"footprint × density ({rules.density_factor})",
                  color="#a0a0a0", edgecolor="black", lw=0.5)

for bar, val in zip(bars_req, required_areas):
    ax.text(bar.get_x() + bar.get_width() / 2, val + max(required_areas) * 0.02,
            f"{val:.1f}", ha="center", fontsize=9, fontweight="bold")
for bar, val in zip(bars_cap, capacities):
    ax.text(bar.get_x() + bar.get_width() / 2, val + max(capacities) * 0.02,
            f"{val:.1f}", ha="center", fontsize=9, color="#333")

# annotate fail bars
for i, (mid, f) in enumerate(zip(chart_fixtures, fail_flags)):
    if f:
        delta = required_areas[i] - capacities[i]
        ax.annotate(f"+{delta:.1f} m² over capacity → AreaGateFailure",
                    xy=(i, required_areas[i]),
                    xytext=(i, required_areas[i] + max(required_areas) * 0.13),
                    ha="center", fontsize=10, color="#e04a3a",
                    fontweight="bold",
                    arrowprops=dict(arrowstyle="->", color="#e04a3a"))

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Area (m²)")
ax.set_title("Stage 02 area gate — required vs capacity (D024 first-pass)")
ax.legend(loc="upper left")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()

chart_path = out_dir / "01_area_budget.png"
fig.savefig(chart_path, dpi=120, bbox_inches="tight")
plt.show()
print("saved:", chart_path.relative_to(ROOT))


## §3 Footprint geometry × Stage 02 result

Each fixture's footprint polygon, axis-aligned bounding box (dashed), and
status colour (blue=Stage 02 pass, red=Stage 02 fail, grey=Stage 01 fail
upstream). Title shows footprint area, capacity, and bbox short side.

In [ ]:
n = len(results)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.4 * nrows))
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

for ax, mid in zip(axes, sorted(results)):
    r = results[mid]
    floor = r["building"].floors[0]
    poly = Polygon(floor.footprint)
    coords = np.array(floor.footprint)

    minx, miny, maxx, maxy = poly.bounds
    bbox_pts = np.array([
        [minx, miny], [maxx, miny], [maxx, maxy], [minx, maxy], [minx, miny]
    ])

    if r["stage01_status"] == "fail":
        color, status_label = "#888", "Stage 01 fail (cardinality)"
    elif r["stage02_status"] == "fail":
        color, status_label = "#e04a3a", f"Stage 02 fail: {r['stage02_failure_type']}"
    else:
        color, status_label = "#4a90d6", "pass"

    ax.fill(coords[:, 0], coords[:, 1], color=color, alpha=0.18)
    closed_x = np.append(coords[:, 0], coords[0, 0])
    closed_y = np.append(coords[:, 1], coords[0, 1])
    ax.plot(closed_x, closed_y, color=color, lw=2)
    ax.plot(bbox_pts[:, 0], bbox_pts[:, 1], "--", color="#666", lw=0.8)

    area_m2 = poly.area / 1_000_000.0
    capacity = area_m2 * rules.density_factor
    short_side = min(maxx - minx, maxy - miny)
    n_required = sum(1 for u in r["instance"].space_units
                     if u.required) if r["instance"] else "—"

    ax.set_title(f"{mid} — {r['file']}", fontsize=10)
    info = (f"area = {area_m2:.1f} m²\n"
            f"capacity = {capacity:.1f} m²\n"
            f"bbox short = {short_side:.0f} mm\n"
            f"required spaces = {n_required}")
    ax.text(0.02, 0.98, info, transform=ax.transAxes,
            fontsize=8, va="top", ha="left", family="monospace",
            bbox=dict(boxstyle="round", facecolor="white",
                      edgecolor="#ccc", alpha=0.85))
    ax.text(0.02, 0.02, status_label, transform=ax.transAxes,
            fontsize=9, va="bottom", color=color, fontweight="bold")
    ax.set_aspect("equal")
    ax.set_xlabel("mm")
    ax.set_ylabel("mm")
    ax.grid(alpha=0.2)

for ax in axes[len(results):]:
    ax.axis("off")

fig.suptitle("Footprint geometry × Stage 02 result", fontsize=13, y=1.0)
fig.tight_layout()

chart_path = out_dir / "02_footprint_geometry.png"
fig.savefig(chart_path, dpi=120, bbox_inches="tight")
plt.show()
print("saved:", chart_path.relative_to(ROOT))


## §4 Required-area breakdown by role

Stacked bars per fixture: how the required-area sum decomposes by role.
The black bracket marks each fixture's gross capacity (footprint ×
density). Tall stacks above the bracket = `AreaGateFailure`.

In [ ]:
roles = ["public", "private", "service", "wet", "hub", "corridor"]
ROLE_COLORS = {
    "public":   "#ffb84d",
    "private":  "#8cc97a",
    "service":  "#b58ad1",
    "wet":      "#9fc8e5",
    "hub":      "#e69191",
    "corridor": "#aaaaaa",
}

stage01_pass_ids = [mid for mid in sorted(results)
                   if results[mid]["stage01_status"] != "fail"]
data = {role: [] for role in roles}
labels_2 = []
caps = []

for mid in stage01_pass_ids:
    r = results[mid]
    inst = r["instance"]
    by_role = {role: 0.0 for role in roles}
    for u in inst.space_units:
        if u.required and u.min_area_m2:
            by_role[u.role] = by_role.get(u.role, 0.0) + u.min_area_m2
    for role in roles:
        data[role].append(by_role[role])
    labels_2.append(mid)

    floor = r["building"].floors[0]
    area_m2 = Polygon(floor.footprint).area / 1_000_000.0
    caps.append(area_m2 * rules.density_factor)

fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(labels_2))
bottom = np.zeros(len(labels_2))
for role in roles:
    heights = np.array(data[role])
    ax.bar(x, heights, bottom=bottom, label=role,
           color=ROLE_COLORS[role], edgecolor="white", linewidth=0.6)
    # numeric annotation per non-zero segment
    for i, h in enumerate(heights):
        if h > 0:
            ax.text(x[i], bottom[i] + h / 2, f"{h:.0f}",
                    ha="center", va="center", fontsize=8, color="#222")
    bottom += heights

ax.scatter(x, caps, marker="_", s=550, lw=3, color="black",
           label=f"capacity (footprint × {rules.density_factor})", zorder=5)

# annotate over-capacity gap
for i, (mid, total, cap) in enumerate(zip(labels_2, bottom, caps)):
    if total > cap:
        ax.annotate(f"{total - cap:+.1f} m²",
                    xy=(i, total), xytext=(i, total + 6),
                    ha="center", fontsize=10, color="#e04a3a",
                    fontweight="bold",
                    arrowprops=dict(arrowstyle="->", color="#e04a3a"))

ax.set_xticks(x)
ax.set_xticklabels(labels_2)
ax.set_ylabel("Area (m²)")
ax.set_title("Required-area decomposition by role")
ax.legend(loc="upper left", ncol=4, fontsize=8)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()

chart_path = out_dir / "03_role_breakdown.png"
fig.savefig(chart_path, dpi=120, bbox_inches="tight")
plt.show()
print("saved:", chart_path.relative_to(ROOT))


## §5 R2 spotlight — first `AreaGateFailure`

`apartment_too_small.json` (4 m × 4 m, 5 required spaces) is Step 06's
showpiece. Plan v2 marked it
`expected_failure: AreaGateFailure / verified_at: Step 06` since the
metadata table was first written. Until §4.6 (`stage02_gate`) landed, that
promise was unverified. Here's the live evidence.

In [ ]:
r2 = results["R2"]
ev = r2["stage02_evidence"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                         gridspec_kw={"width_ratios": [1, 1.6]})

# (a) footprint
ax_g = axes[0]
floor = r2["building"].floors[0]
poly = Polygon(floor.footprint)
coords = np.array(floor.footprint)
ax_g.fill(coords[:, 0], coords[:, 1], color="#e04a3a", alpha=0.25)
ax_g.plot(np.append(coords[:, 0], coords[0, 0]),
          np.append(coords[:, 1], coords[0, 1]),
          color="#e04a3a", lw=2.5)
ax_g.set_aspect("equal")
ax_g.set_title("R2 footprint (4 m × 4 m = 16 m²)")
ax_g.set_xlabel("mm")
ax_g.set_ylabel("mm")
ax_g.grid(alpha=0.3)

# (b) area numbers
ax_b = axes[1]
items = ["required\n(Σ min_area)",
         f"capacity\n(footprint × {rules.density_factor})",
         "footprint\n(gross)"]
values = [ev["total_required_area_m2"],
          ev["usable_capacity_m2"],
          ev["footprint_area_m2"]]
colors = ["#e04a3a", "#4a90d6", "#cccccc"]
bars = ax_b.bar(items, values, color=colors, edgecolor="black", lw=0.8)
for bar, v in zip(bars, values):
    ax_b.text(bar.get_x() + bar.get_width() / 2, v + 0.5,
              f"{v:.1f} m²", ha="center", fontsize=12, fontweight="bold")
ax_b.set_ylabel("Area (m²)")
ax_b.set_title(f"AreaGateFailure — `{r2['stage02_failure_type']}`")
ax_b.text(0.5, -0.18,
          f"Σ required {ev['total_required_area_m2']:.1f} m² > "
          f"capacity {ev['usable_capacity_m2']:.1f} m² "
          f"({ev['required_space_count']} required spaces)",
          transform=ax_b.transAxes, ha="center", fontsize=11,
          color="#444")
ax_b.grid(axis="y", alpha=0.3)

fig.suptitle("R2 spotlight — first live `AreaGateFailure`", fontsize=13)
fig.tight_layout()

chart_path = out_dir / "04_r2_spotlight.png"
fig.savefig(chart_path, dpi=120, bbox_inches="tight")
plt.show()

print("R2 stage02 failure evidence:")
for k, v in ev.items():
    print(f"  {k}: {v}")
print("saved:", chart_path.relative_to(ROOT))


## §6 D023 demo — required-only cardinality

Optional spaces (`required=False`) must NOT satisfy `min_cardinality`. This
synthetic fixture has only one optional bedroom; Stage 01 should fail with
`program_cardinality_insufficient` for `private`.

This corner used to be a silent bug (second external review #1 — D023
formalized the policy + Stage 01 enforces it via
`Counter(u.role for u in space_units if u.required)`).

In [ ]:
synthetic = BuildingInput(
    target_type="apartment",
    program_request=ProgramRequest(spaces=[
        SpaceUnitSpec(name="living", role="public"),
        SpaceUnitSpec(name="study", role="private", required=False),
        SpaceUnitSpec(name="bathroom", role="wet"),
    ]),
)

print("Synthetic ProgramRequest:")
for u in synthetic.program_request.spaces:
    flag = "(optional)" if not u.required else ""
    print(f"  {u.name:<10}  role={u.role:<8}  {flag}")
print()

try:
    stage01_program.run(synthetic, adapter=adapter)
    print("UNEXPECTED: cardinality passed (D023 broken)")
except ProgramInstantiationFailure as e:
    print(f"Expected ProgramInstantiationFailure raised:")
    print(f"  failure_type:    {e.failure.failure_type}")
    print(f"  affected_space:  {e.failure.affected_space}")
    print(f"  detected_stage:  {e.failure.detected_stage}")
    print(f"  evidence:        {e.failure.evidence}")
    print(f"  diagnosis:       {e.failure.diagnosis}")


---

## Notes / next step

### Step 06 산출물

- `ProgramRequest` typed dataclass (slim — `spaces` only)
- `Role` Literal (6 canonical roles, strict)
- `TargetRules` dataclass + `data/target_rules/apartment.json` external config
- **Single generic `TargetAdapter`** (S06-D22, no per-typology classes)
- `DomainGateFailure` hierarchy (`AreaGateFailure` / `DimGateFailure` /
  `AccessSchemaFailure`)
- `proto3.constraints.gates` 4 pure functions (3 active in Stage 02 + access
  dormant for Step 09–10)
- **Required-only** cardinality / area summation (D023)
- `RunConfig.__post_init__` value validation
- `atom_inclusion_threshold` wiring (was a dead config)
- Fail-loud sweep — `palette` / `render` / Stage 01 type guard

### External pipeline integration

`TargetAdapter(rules_path=external.json)` — proto3 코드 변경 0. New typology
within the existing 5 × 6 grid (5 `TargetType` × 6 `Role`) = JSON file +
1-line dispatch entry in `stage00_load._DEFAULT_ADAPTERS`. Outside the grid
needs a 1-line `Literal` extension alongside.

### Next Step (Step 07 — Region/Atom Decomposition)

- Stage 00 normalize 책임 확장 (Plan Def-6)
- Decomposition → `RegionSet` / `AtomSet` 본격 매핑 (M2 layered approach)
- v12 zoning port (Plan Def-13)
- Hole-aware decompose / schema (Plan Def-11) — `decompose.run` + `to_schema`
  + `geometry.py` 모두 single-ring 전제
- Stage 09–10 에서 L2 strategy registry (S06-D22) 도입 — first concrete
  `apartment` strategy lands then.
